In [ ]:
%matplotlib ipympl
from pathlib import Path

from matplotlib import pyplot as plt
from ipywidgets import widgets

from topepan import plot as tpp

In [ ]:
# The run to browse, as either its output directory or -- for CrunchTope -- its input deck.
# CrunchTope records its output times in the deck and nowhere else, so giving the deck gets both the
# output beside it and the times; MIN3P stamps each snapshot with its own time, so pointing at the
# directory is enough. Edit and re-run from here down.
RUN = '/path/to/your/run/model.in'

# For CrunchTope the times come from the deck rather than from a .tec header: MineralPercent, which
# the alternative reads, only exists in 2.10 and later, so an older build writes no such file.
catList, max_time, times = tpp.open_run(RUN)

In [ ]:
# Snapshot numbering is not the same in both codes, and under MIN3P it is not even the same for
# every category: CrunchTope writes every quantity at every output time numbered from 1, while MIN3P
# numbers from 0 -- snapshot 0 being the initial state -- and writes the flow field far fewer times
# than the chemistry. So the slider is ranged per category rather than once.
snapshot_cache = {}


def snapshots_for(cat):
    if cat not in snapshot_cache:
        indices = tpp.snapshot_indices(cat) or list(range(1, max_time + 1))
        # MIN3P stamps each snapshot with its own time; CrunchTope states them once, in the deck.
        stamps = tpp.output_times(file_cat=cat) if tpp.simulator() == 'min3p' else times
        snapshot_cache[cat] = (indices, stamps)

    return snapshot_cache[cat]


file_cat = widgets.ToggleButtons(options=catList, description='Output type')
_indices = snapshots_for(catList[0])[0]
time_slider = widgets.IntSlider(value=_indices[0], min=_indices[0], max=_indices[-1], step=1,
                                description='Snapshot')
plot_var = widgets.Select(options=tpp.read_tecplot(file_cat.value, time_slider.value)[1][3:],
                          description='Variable')
# X down the y axis is the depth convention; X across the x axis reads better for a flow path.
orientation = widgets.ToggleButtons(
    options=[('distance on y axis (depth)', True), ('distance on x axis', False)],
    value=True, description='Orientation')
log_scale = widgets.Checkbox(value=False, description='log10')


def update_plot_vars(*args):
    indices, _ = snapshots_for(file_cat.value)
    # Widened before it is narrowed: setting a min above the current max, or a max below the
    # current min, is rejected outright rather than clamped.
    time_slider.min = 0
    time_slider.max = max(indices)
    time_slider.min = min(indices)
    plot_var.options = tpp.read_tecplot(file_cat.value, time_slider.value)[1][3:]


def update_plot(time, file_cat, plot_var, orientation, log10):
    if plot_var is None:
        return
    df, column_headers = tpp.read_tecplot(file_cat, time)
    # Ranged over every snapshot, so the axis does not jump about as the slider moves.
    lower, upper = tpp.plot_var_range(time_slider.max, file_cat, plot_var)

    # Pad the value axis away from the data, and give a flat profile a range it can be seen in.
    if upper > 0:
        upper = upper * 1.02
    elif upper < 0:
        upper = upper * 0.98
    else:
        upper = -lower

    if lower > 0:
        lower = lower * 0.98
    elif lower < 0:
        lower = lower * 1.02
    else:
        lower = -upper

    if upper == lower:
        upper = upper + 1
        lower = lower - 1

    # Whichever axis the column runs down: MIN3P's dissolution benchmark is a Z column, and
    # plotting it against a constant X would draw a vertical line instead of a profile.
    space = tpp.profile_axis(df)
    tpp.draw_profile(ax, df[space], df[plot_var], vertical=orientation, lower=lower, upper=upper,
                     value_label=plot_var, log10=log10, space_label=space)
    # Which snapshot is on screen, since the slider only gives its index. A title rather than an
    # axis label because this is a browser, not a figure headed for a paper.
    indices, stamps = snapshots_for(file_cat)
    position = indices.index(time) if time in indices else None
    stamp = stamps[position] if position is not None and position < len(stamps) else None

    # MIN3P's snapshot 0 is the initial state and carries no time, so it is named rather than dated.
    if stamp is not None and stamp == stamp:
        ax.set_title(f'time = {stamp:g}')
    else:
        ax.set_title(f'snapshot {time}')
    fig.canvas.draw_idle()


file_cat.observe(update_plot_vars, 'value')
fig, ax, line = tpp.initialise1D(catList[0])
widgets.interact(update_plot, file_cat=file_cat, time=time_slider, plot_var=plot_var,
                 orientation=orientation, log10=log_scale)

In [ ]:
plt.close('all')